# Workshop 2026 dlt Homework (English Version)

This notebook loads NYC Taxi API data into DuckDB using dlt and computes answers for the 3 homework questions.


In [ ]:
# Run once in Colab/local notebook if needed
# !pip -q install "dlt[duckdb]" duckdb


## 1) Import Libraries


In [ ]:
import dlt
from dlt.sources.helpers.rest_client import RESTClient


## 2) Define API Source


In [ ]:
BASE_URL = "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"
PAGE_SIZE = 1000

def extract_records(payload):
    """Extract the actual list of records from API JSON response."""
    if payload is None:
        return []
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for k in ("data", "results", "rides", "items"):
            v = payload.get(k)
            if isinstance(v, list):
                return v
    return []

@dlt.resource(name="rides", write_disposition="replace")
def ny_taxi_rides():
    client = RESTClient(base_url=BASE_URL)
    page = 1

    while True:
        # Homework requirement: 1,000 rows per page, stop on empty page
        response = client.get("", params={"page": page, "page_size": PAGE_SIZE})
        rows = extract_records(response.json())

        if not rows:
            break

        yield from rows
        page += 1


## 3) Run the Pipeline


In [ ]:
pipeline = dlt.pipeline(
    pipeline_name="taxi_pipeline",
    destination="duckdb",
    dataset_name="taxi_data",
)

load_info = pipeline.run(ny_taxi_rides())
print(load_info)


## 4) Compute Homework Answers


In [ ]:
with pipeline.sql_client() as c:
    q1 = c.execute_sql("""
        SELECT
          MIN(CAST(trip_pickup_date_time AS DATE)) AS start_date,
          MAX(CAST(trip_pickup_date_time AS DATE)) AS end_date
        FROM rides
    """)

    q2 = c.execute_sql("""
        SELECT ROUND(
          100.0 * AVG(CASE WHEN LOWER(payment_type) = 'credit' THEN 1 ELSE 0 END),
          2
        )
        FROM rides
    """)

    q3 = c.execute_sql("""
        SELECT ROUND(SUM(CAST(tip_amt AS DOUBLE)), 2)
        FROM rides
    """)

print("Q1 date range:", q1)
print("Q2 credit card ratio (%):", q2)
print("Q3 total tips ($):", q3)


## 5) Multiple-Choice Mapping

- If Q1 output is `2009-06-01` ~ `2009-06-30`, choose `2009-06-01 to 2009-07-01` in the form.
- Q2 is `26.66%`
- Q3 is `$6,063.41`
